In [7]:
import pandas as pd

csv_path = r'C:\Users\vikas\Documents\ProMain1\Data\final_data.csv'

df = pd.read_csv(csv_path)

print(df.head())

          MMSI                  time        Lat         Lon   SOG         COG  \
0  316050336.0  2024-01-01T00:00:18Z  49.305809 -124.121277  15.5  165.800003   
1  367308416.0  2024-01-01T00:00:41Z  47.538921 -122.647293   0.0  360.000000   
2  367662464.0  2024-01-01T00:01:14Z  38.833618  -82.151543   3.1   70.099998   
3  303332992.0  2024-01-01T00:01:18Z  27.731569  -90.208649   9.2  179.899994   
4  368203648.0  2024-01-01T00:01:21Z  38.933510  -82.110649   5.2   20.700001   

   Heading  VesselType  Status  Length  ...  Cargo  temperature_2m  \
0    165.0        37.0     NaN    19.0  ...    NaN       21.920000   
1    511.0        37.0     NaN    14.0  ...    NaN       23.160000   
2    511.0        52.0     0.0    22.0  ...   52.0       22.040001   
3    187.0        70.0     0.0    80.0  ...   70.0       15.860000   
4     14.0        31.0    12.0    37.0  ...   31.0       15.900000   

   relative_humidity   rain  snowfall  sea_level_pressure  total_cloud_cover  \
0          6

In [8]:
#remove duplicates

print("Before:", len(df))
df = df.drop_duplicates()
print("After:", len(df))

Before: 10184763
After: 10184763


In [9]:
#fixing date and time

df['time'] = pd.to_datetime(df['time'])
df['date'] = df['time'].dt.strftime('%d-%m-%Y')
df['month'] = df['time'].dt.strftime('%b')
df['time'] = df['time'].dt.strftime('%H:%M:%S')

print(df[['date', 'month', 'time']].head())

         date month      time
0  01-01-2024   Jan  00:00:18
1  01-01-2024   Jan  00:00:41
2  01-01-2024   Jan  00:01:14
3  01-01-2024   Jan  00:01:18
4  01-01-2024   Jan  00:01:21


In [10]:
#drop row and changing data

df = df.drop(columns=['Heading'])

vessel_mapping = {
    31.0: 'Tug',
    37.0: 'Sailing',
    52.0: 'Special Craft',
    70.0: 'Cargo',
    31.0: 'Fishing',
}

df['VesselType'] = df['VesselType'].map(vessel_mapping).fillna('Other / Unknown')

print(df[['VesselType']].head())

      VesselType
0        Sailing
1        Sailing
2  Special Craft
3          Cargo
4        Fishing


In [11]:
#drop column

df = df.drop(columns=['Status'])
print(df.columns)

Index(['MMSI', 'time', 'Lat', 'Lon', 'SOG', 'COG', 'VesselType', 'Length',
       'Width', 'Draft', 'Cargo', 'temperature_2m', 'relative_humidity',
       'rain', 'snowfall', 'sea_level_pressure', 'total_cloud_cover',
       'wind_speed_10m', 'wind_direction_10m', 'soil_temperature_0_to_7cm',
       'date', 'month'],
      dtype='str')


In [12]:
#removing decimal

df = df.drop(columns=['Width'], errors='ignore')

df['Length'] = pd.to_numeric(df['Length'], errors='coerce').astype('Int64')

print(df[['Length']].head())

   Length
0      19
1      14
2      22
3      80
4      37


In [13]:
#removing decimal

df = df.drop(columns=['Cargo'], errors='ignore')

df['temperature_2m'] = df['temperature_2m'].round(2)
df['relative_humidity'] = df['relative_humidity'].round(2)

print(df[['temperature_2m', 'relative_humidity']].head())

   temperature_2m  relative_humidity
0           21.92              68.56
1           23.16              73.15
2           22.04              56.16
3           15.86              93.51
4           15.90              68.45


In [14]:
#fixing columns

df['snowfall'] = pd.to_numeric(df['snowfall'], errors='coerce').fillna(0)
df['snowfall'] = df['snowfall'].apply(lambda x: 'no' if x == 0 else 'yes')

cols_to_round = ['sea_level_pressure', 'total_cloud_cover', 'wind_speed_10m']
for col in cols_to_round:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').round(2)

df = df.drop(columns=['wind_direction_10m', 'soil_temperature_0_to_7cm'], errors='ignore')

print(df[['snowfall', 'sea_level_pressure', 'total_cloud_cover', 'wind_speed_10m']].head())

  snowfall  sea_level_pressure  total_cloud_cover  wind_speed_10m
0       no              984.03             100.00           53.90
1       no              999.99             100.00           35.36
2       no             1016.75               3.84            9.05
3       no             1002.80              92.19           26.84
4       no             1019.25               8.81            8.99


In [15]:
#adding new columns

dt_series = pd.to_datetime(df['time'], format='mixed', errors='coerce')

df['hour'] = dt_series.dt.hour
df['day_of_week'] = dt_series.dt.day_name()

def categorize_wind(speed):
    if pd.isna(speed):
        return 'Unknown'
    elif speed < 10:
        return 'Calm'
    elif speed < 25:
        return 'Moderate'
    elif speed < 40:
        return 'Strong'
    else:
        return 'Stormy'

df['wind_category'] = df['wind_speed_10m'].apply(categorize_wind)

print(df[['time', 'hour', 'day_of_week', 'wind_speed_10m', 'wind_category']].head())

       time  hour day_of_week  wind_speed_10m wind_category
0  00:00:18     0      Monday           53.90        Stormy
1  00:00:41     0      Monday           35.36        Strong
2  00:01:14     0      Monday            9.05          Calm
3  00:01:18     0      Monday           26.84        Strong
4  00:01:21     0      Monday            8.99          Calm


In [16]:
print(df.head())

          MMSI      time        Lat         Lon   SOG         COG  \
0  316050336.0  00:00:18  49.305809 -124.121277  15.5  165.800003   
1  367308416.0  00:00:41  47.538921 -122.647293   0.0  360.000000   
2  367662464.0  00:01:14  38.833618  -82.151543   3.1   70.099998   
3  303332992.0  00:01:18  27.731569  -90.208649   9.2  179.899994   
4  368203648.0  00:01:21  38.933510  -82.110649   5.2   20.700001   

      VesselType  Length  Draft  temperature_2m  ...   rain  snowfall  \
0        Sailing      19    NaN           21.92  ...  26.60        no   
1        Sailing      14    NaN           23.16  ...   0.14        no   
2  Special Craft      22    2.6           22.04  ...   0.00        no   
3          Cargo      80    4.0           15.86  ...   1.97        no   
4        Fishing      37    3.0           15.90  ...   0.00        no   

  sea_level_pressure  total_cloud_cover  wind_speed_10m        date month  \
0             984.03             100.00           53.90  01-01-2024  

In [18]:
df.to_csv(
    r'C:\Users\vikas\Documents\ProMain1\Data\Final_Maritime_Clean_Data.csv',
    index=False
)
print("Final Maritime Clean Data saved successfully!")

Final Maritime Clean Data saved successfully!
